# E v7 held-out: NQ Diagnosis-3216 pool, run once

300 NQ samples E has never seen (120 utilisation, 35 accessibility, 100 correct, 45 insufficiency
by offline annotation; labels are not in the bundle). Strategy `short_answer_quote`, frozen in
docs/ANSWER_REPAIR_V7.md before this cohort existed. One Qwen NF4 call per sample, 384 tokens.
Run every cell once, download the ZIP, and report the result whatever it is. No reruns, no tuning.


In [ ]:
import json, os, sys, subprocess, hashlib, tarfile, shutil
from pathlib import Path
import torch
assert sys.version_info >= (3, 11), 'Python 3.11+ required'
assert torch.cuda.is_available(), 'Select a GPU runtime'
WORK = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('/content')
EXPECTED_SHA256 = '6f27757935128f3e04e2881e1fb832885b10d45ef55316fa425b889950651bde'
BUNDLE_NAME = 'answer_repair_v7_heldout3216_gpu'
OUTPUT = WORK / (BUNDLE_NAME.removesuffix('_gpu') + '_outputs')
OUTPUT.mkdir(parents=True, exist_ok=True)
print('GPU:', torch.cuda.get_device_name(0), 'CUDA:', torch.version.cuda)


In [ ]:
# Kaggle may unpack the uploaded archive before attaching it.
EXPECTED_INDEX_SHA256 = '05e42ff1904bdd8364c9b2969762787cf568f8cce5539ee2ed4390a5c3e92c06'
archive = WORK / (BUNDLE_NAME + '.tar.gz')
extract_root = WORK / ('e_reviewed_' + EXPECTED_SHA256[:12])
BUNDLE = extract_root / BUNDLE_NAME
transport = 'archive_sha256'
if not archive.exists() and Path('/kaggle/input').exists():
    candidates = list(Path('/kaggle/input').rglob(archive.name))
    if candidates:
        assert len(candidates) == 1, '实验归档必须唯一'
        archive = candidates[0]
    else:
        indexes = [p for p in Path('/kaggle/input').rglob('bundle_checksums.json')
                   if p.parent.name == BUNDLE_NAME]
        assert len(indexes) == 1, '请附加唯一的冻结实验包输入'
        assert hashlib.sha256(indexes[0].read_bytes()).hexdigest() == EXPECTED_INDEX_SHA256
        source = indexes[0].parent
        source_checksums = json.loads(indexes[0].read_text())
        assert all(hashlib.sha256((source / p).read_bytes()).hexdigest() == h
                   for p,h in source_checksums.items()), '已解压输入文件哈希不一致'
        if not BUNDLE.exists():
            shutil.copytree(source, BUNDLE)
        transport = 'pinned_checksum_index_and_all_files'
elif not archive.exists():
    from google.colab import files
    uploaded = files.upload()
    assert archive.name in uploaded, '请选择配套实验包'
    archive.write_bytes(uploaded[archive.name])
if transport == 'archive_sha256':
    assert hashlib.sha256(archive.read_bytes()).hexdigest() == EXPECTED_SHA256
    if not BUNDLE.exists():
        extract_root.mkdir(parents=True, exist_ok=True)
        with tarfile.open(archive) as handle:
            handle.extractall(extract_root, filter='data')
assert hashlib.sha256((BUNDLE / 'bundle_checksums.json').read_bytes()).hexdigest() == EXPECTED_INDEX_SHA256
checksums = json.loads((BUNDLE / 'bundle_checksums.json').read_text())
assert all(hashlib.sha256((BUNDLE / p).read_bytes()).hexdigest() == h for p,h in checksums.items())
manifest = json.loads((BUNDLE / 'manifest.json').read_text())
integrity = {'transport': transport, 'expected_archive_sha256': EXPECTED_SHA256,
             'checksum_index_sha256': EXPECTED_INDEX_SHA256,
             'verified_files': len(checksums), 'all_files_verified': True}
(OUTPUT / 'input_integrity.json').write_text(json.dumps(integrity, indent=2))
print('输入校验通过:', integrity)
print({ds: r['groups'] for ds,r in manifest.get('datasets', {}).items()} or manifest.get('samples'))


In [ ]:
def run_logged(argv, name):
    with (OUTPUT / name).open('a', encoding='utf-8') as log:
        log.write('COMMAND ' + json.dumps(argv) + '\n'); log.flush()
        process = subprocess.Popen(argv, cwd=BUNDLE, env={**os.environ, 'PYTHONUNBUFFERED': '1'},
                                   stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
        try:
            for line in process.stdout:
                print(line, end='', flush=True); log.write(line); log.flush()
            status = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if status: raise subprocess.CalledProcessError(status, argv)

run_logged([sys.executable, '-m', 'pip', 'install', '-r', 'requirements-answer-repair.txt'], 'install.log')
run_logged([sys.executable, 'scripts/check_answer_repair_environment.py', '--require-cuda',
            '--out', str(OUTPUT / 'environment.json')], 'environment.log')
run_logged([sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'], 'tests.log')
for path in ('manifest.json', 'bundle_checksums.json'):
    shutil.copy2(BUNDLE / path, OUTPUT / path)
arm = manifest['protocol'][0]
RESULT = OUTPUT / 'nq_short_answer_quote_heldout3216.jsonl'
ARGV = [sys.executable, '-u', 'scripts/run_answer_repair.py', '--dataset', 'nq',
        '--c1', str(BUNDLE / 'inputs/nq_heldout3216_c1.jsonl'), '--all',
        '--strategy', arm['strategy'], '--max-new-tokens', str(arm['max_new_tokens']), '--out', str(RESULT)]
run_logged(ARGV + ['--check-only'], 'nq_short_answer_quote_heldout3216_preflight.log')


## Run once
Resumable within the same runtime if interrupted; never rerun after completion.


In [ ]:
run_logged(ARGV, 'nq_short_answer_quote_heldout3216_run.log')


In [ ]:
export = shutil.make_archive(str(WORK / (OUTPUT.name + '_export')), 'zip', OUTPUT)
print('Result archive:', export)
print('SHA256:', hashlib.sha256(Path(export).read_bytes()).hexdigest())
if not Path('/kaggle/working').exists():
    from google.colab import files
    files.download(export)
